In [24]:
import pandas as pd
import seaborn as sns
import matplotlib
import requests
import zipfile
import json
import io
import glob
import os
import sys

from pathlib import Path

# Racine du repository data
PROJECT_ROOT = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(PROJECT_ROOT))

# Configuration des chemins
from lib.config import AN_BRONZE
from lib.config import AN_SILVER

In [25]:
"""
input_file = AN_BRONZE / "scrutins.json.zip"

print(f"Lecture du fichier : {input_file}")
"""

'\ninput_file = AN_BRONZE / "scrutins.json.zip"\n\nprint(f"Lecture du fichier : {input_file}")\n'

In [26]:
url_votes = "http://data.assemblee-nationale.fr/static/openData/repository/17/loi/scrutins/Scrutins.json.zip"

response = requests.get(url_votes)
response.raise_for_status()

zip_file = zipfile.ZipFile(io.BytesIO(response.content))

print(len(zip_file.namelist()), "fichiers dans le ZIP")

9335 fichiers dans le ZIP


In [27]:
zip_file.namelist()[:5]

['json/VTANR5L17V2657.json',
 'json/VTANR5L17V6722.json',
 'json/VTANR5L17V5293.json',
 'json/VTANR5L17V3985.json',
 'json/VTANR5L17V1144.json']

In [28]:
scrutins = []
votes_individuels = []

In [29]:
for fichier in zip_file.namelist():

    if not fichier.endswith(".json"):
        continue

    with zip_file.open(fichier) as f:
        data = json.load(f)

    scrutin = data["scrutin"]

    uid_scrutin = scrutin["uid"]

    # -------------------------
    # TABLE SCRUTINS
    # -------------------------

    synthese = scrutin.get("syntheseVote", {})
    decompte = synthese.get("decompte", {})

    scrutins.append({
        "uid": uid_scrutin,
        "numero": scrutin.get("numero"),
        "legislature": scrutin.get("legislature"),
        "date_scrutin": scrutin.get("dateScrutin"),
        "type_vote": scrutin.get("typeVote", {}).get("libelleTypeVote"),
        "sort": scrutin.get("sort", {}).get("code"),
        "titre": scrutin.get("titre"),

        "nombre_votants": synthese.get("nombreVotants"),
        "suffrages_exprimes": synthese.get("suffragesExprimes"),

        "pour": decompte.get("pour"),
        "contre": decompte.get("contre"),
        "abstentions": decompte.get("abstentions"),
        "non_votants": decompte.get("nonVotants")
    })

    # -------------------------
    # TABLE VOTES INDIVIDUELS
    # -------------------------

    ventilation = scrutin.get("ventilationVotes", {})
    organe = ventilation.get("organe", {})
    groupes = organe.get("groupes", {}).get("groupe", [])

    # Un seul groupe peut éventuellement être représenté par un dict
    if isinstance(groupes, dict):
        groupes = [groupes]

    for groupe in groupes:

        organe_groupe = groupe.get("organeRef")

        vote = groupe.get("vote", {})
        nominatif = vote.get("decompteNominatif", {})

        positions = {
            "pour": nominatif.get("pours"),
            "contre": nominatif.get("contres"),
            "abstention": nominatif.get("abstentions"),
            "non_votant": nominatif.get("nonVotants")
        }

        for position, bloc in positions.items():

            if bloc is None:
                continue

            votants = bloc.get("votant")

            if votants is None:
                continue

            # Un seul votant = dict
            if isinstance(votants, dict):
                votants = [votants]

            for votant in votants:

                votes_individuels.append({
                    "uid_scrutin": uid_scrutin,
                    "organe_groupe": organe_groupe,
                    "acteur_ref": votant.get("acteurRef"),
                    "mandat_ref": votant.get("mandatRef"),
                    "position": position,
                    "par_delegation": votant.get("parDelegation"),
                    "num_place": votant.get("numPlace"),
                    "cause_position_vote": votant.get("causePositionVote")
                })

In [30]:
df_scrutins = pd.DataFrame(scrutins)

df_votes_individuels = pd.DataFrame(votes_individuels)

In [36]:
df_scrutins.head()

,uid,numero,legislature,date_scrutin,type_vote,sort,titre,nombre_votants,suffrages_exprimes,pour,contre,abstentions,non_votants
0,VTANR5L17V2657,2657,17,2025-06-24,scrutin public ordinaire,adopté,l'article 22 (examen prioritaire) du projet de...,144,144,121,23,0,2
1,VTANR5L17V6722,6722,17,2026-05-18,scrutin public ordinaire,adopté,l'article 26 du projet de loi actualisant la p...,89,89,89,0,0,1
2,VTANR5L17V5293,5293,17,2026-02-02,scrutin public ordinaire,rejeté,l'amendement n° 15 de M. Rancoule à l'article ...,97,96,21,75,1,3
3,VTANR5L17V3985,3985,17,2025-11-19,scrutin public ordinaire,rejeté,l'amendement n° 2194 de Mme Sylvie Bonnet aprè...,210,199,14,185,11,2
4,VTANR5L17V1144,1144,17,2025-03-24,scrutin public ordinaire,adopté,l'amendement n° 585 de M. Pauget à l'article 1...,102,93,63,30,9,2


In [35]:
df_votes_individuels.head()

,uid_scrutin,organe_groupe,acteur_ref,mandat_ref,position,par_delegation,num_place,cause_position_vote
0,VTANR5L17V2657,PO845401,PA842279,PM843779,pour,false,026,None
1,VTANR5L17V2657,PO845401,PA793238,PM842426,pour,false,073,None
2,VTANR5L17V2657,PO845401,PA793158,PM840258,pour,false,075,None
3,VTANR5L17V2657,PO845401,PA794434,PM842990,pour,false,002,None
4,VTANR5L17V2657,PO845401,PA720668,PM840342,pour,false,060,None


In [32]:
output_file_scrutins = AN_SILVER / "scrutins.parquet"
output_file_scrutins.parent.mkdir(parents=True, exist_ok=True)

df_scrutins.to_parquet(output_file_scrutins, index=False)

print(f"Fichier enregistré : {output_file_scrutins}")

Fichier enregistré : \srv\popolitics\data\AN\silver\scrutins.parquet


In [33]:
output_file_votes_individuels = AN_SILVER / "votes_individuels.parquet"
output_file_votes_individuels.parent.mkdir(parents=True, exist_ok=True)

df_votes_individuels.to_parquet(output_file_votes_individuels, index=False)

print(f"Fichier enregistré : {output_file_votes_individuels}")

Fichier enregistré : \srv\popolitics\data\AN\silver\votes_individuels.parquet
